<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import Counter
import numpy as np
import pandas as pd


dataset = [
    [2.0, 45.0, 0],
    [2.5, 50.0, 0],
    [3.0, 48.0, 0],
    [3.5, 55.0, 0],
    [4.0, 52.0, 0],
    [4.5, 60.0, 0],
    [5.0, 58.0, 0],
    [5.2, 62.0, 0],
    [5.6, 66.0, 1],
    [6.0, 68.0, 1],
    [6.5, 72.0, 1],
    [7.0, 75.0, 1],
    [7.5, 80.0, 1],
    [8.0, 82.0, 1],
    [8.5, 85.0, 1],
    [9.0, 90.0, 1],
    [9.5, 92.0, 1],
    [4.8, 61.0, 0],
    [6.2, 70.0, 1],
    [7.8, 84.0, 1],
]

df = pd.DataFrame(dataset, columns=["Study_Hours", "Attendance_Pct", "Pass"])

X = df[["Study_Hours", "Attendance_Pct"]].values.astype(float)
y = df["Pass"].values.astype(int)

test_idx = [3, 6, 8, 11, 14, 17]
train_mask = np.ones(len(X), dtype=bool)
train_mask[test_idx] = False

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_idx], y[test_idx]

mean = X_train.mean(axis=0)
std = np.where(X_train.std(axis=0) == 0, 1.0, X_train.std(axis=0))
X_tr_s = (X_train - mean) / std
X_te_s = (X_test - mean) / std


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def train_logistic(X_s, y_bin, lr=0.2, epochs=2000, seed=42):
    rng = np.random.default_rng(seed)
    w = rng.normal(0.0, 0.1, size=(X_s.shape[1], 1))
    b = 0.0
    y_col = y_bin.reshape(-1, 1).astype(float)
    for _ in range(epochs):
        pred = sigmoid(X_s @ w + b)
        m = len(y_bin)
        w -= lr * ((X_s.T @ (pred - y_col)) / m)
        b -= lr * np.mean(pred - y_col)
    return w, b


w_lr, b_lr = train_logistic(X_tr_s, y_train)
pred_lr = (sigmoid(X_te_s @ w_lr + b_lr).flatten() >= 0.5).astype(int)


def predict_knn(X_tr, y_tr, X_te, k=3):
    out = []
    for row in X_te:
        dists = np.linalg.norm(X_tr - row, axis=1)
        nearest = y_tr[np.argsort(dists)[:k]]
        out.append(Counter(nearest).most_common(1)[0][0])
    return np.array(out, dtype=int)


pred_knn = predict_knn(X_tr_s, y_train, X_te_s, k=3)


def predict_nb(X_tr, y_tr, X_te):
    classes = np.unique(y_tr)
    priors = {c: np.mean(y_tr == c) for c in classes}
    stats = {}
    for c in classes:
        sub = X_tr[y_tr == c]
        stats[c] = (np.mean(sub, axis=0), np.var(sub, axis=0) + 1e-6)
    preds = []
    for row in X_te:
        scores = {}
        for c in classes:
            m_c, v_c = stats[c]
            ll = -0.5 * np.sum(np.log(2.0 * np.pi * v_c) + ((row - m_c) ** 2) / v_c)
            scores[c] = np.log(priors[c]) + ll
        preds.append(max(scores, key=scores.get))
    return np.array(preds, dtype=int)


pred_nb = predict_nb(X_train, y_train, X_test)


def train_perceptron(X_s, y_bin, lr=0.1, epochs=500, seed=42):
    rng = np.random.default_rng(seed)
    w = rng.normal(0.0, 0.1, size=X_s.shape[1])
    b = 0.0
    for _ in range(epochs):
        for x_i, y_i in zip(X_s, y_bin):
            y_pred = 1 if (np.dot(x_i, w) + b) >= 0 else 0
            err = y_i - y_pred
            w += lr * err * x_i
            b += lr * err
    return w, b


w_p, b_p = train_perceptron(X_tr_s, y_train)
pred_perc = np.where((X_te_s @ w_p + b_p) >= 0, 1, 0).astype(int)


def eval_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    acc = (tp + tn) / len(y_true) * 100
    prec = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1, np.array([[tn, fp], [fn, tp]])


models = {
    "Logistic Regression": pred_lr,
    "K-NN (k=3)": pred_knn,
    "Gaussian Naive Bayes": pred_nb,
    "Perceptron": pred_perc,
}

summary_rows = []
for name, y_p in models.items():
    acc, prec, rec, f1, cm = eval_metrics(y_test, y_p)
    summary_rows.append({
        "Model": name,
        "Accuracy (%)": round(acc, 2),
        "Precision (%)": round(prec, 2),
        "Recall (%)": round(rec, 2),
        "F1-Score (%)": round(f1, 2),
    })
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.2f}% | Precision: {prec:.2f}% | Recall: {rec:.2f}% | F1: {f1:.2f}%")
    print("Confusion Matrix [TN, FP; FN, TP]:")
    print(cm)
    print()

summary_df = pd.DataFrame(summary_rows)
print("=== Comparison Summary ===")
print(summary_df.to_string(index=False))

=== Logistic Regression ===
Accuracy: 100.00% | Precision: 100.00% | Recall: 100.00% | F1: 100.00%
Confusion Matrix [TN, FP; FN, TP]:
[[3 0]
 [0 3]]

=== K-NN (k=3) ===
Accuracy: 100.00% | Precision: 100.00% | Recall: 100.00% | F1: 100.00%
Confusion Matrix [TN, FP; FN, TP]:
[[3 0]
 [0 3]]

=== Gaussian Naive Bayes ===
Accuracy: 100.00% | Precision: 100.00% | Recall: 100.00% | F1: 100.00%
Confusion Matrix [TN, FP; FN, TP]:
[[3 0]
 [0 3]]

=== Perceptron ===
Accuracy: 83.33% | Precision: 100.00% | Recall: 66.67% | F1: 80.00%
Confusion Matrix [TN, FP; FN, TP]:
[[3 0]
 [1 2]]

=== Comparison Summary ===
               Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)
 Logistic Regression        100.00          100.0      100.00         100.0
          K-NN (k=3)        100.00          100.0      100.00         100.0
Gaussian Naive Bayes        100.00          100.0      100.00         100.0
          Perceptron         83.33          100.0       66.67          80.0
